# Lesson 23: Fairness and Causal Inference

## Opening Story: Algorithmic Bias

In 2019, a study found that a healthcare algorithm used on millions of patients was biased against Black patients. The algorithm used healthcare costs as a proxy for health needs, but because Black patients had less access to healthcare, they had lower costs even when equally sick.

This is a causal problem: the algorithm confounded cost with need, leading to unfair treatment. Causal inference provides tools to understand and address algorithmic bias.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Define fairness criteria in causal terms
2. Explain the difference between observational and causal fairness
3. Implement fairness-aware algorithms
4. Conduct causal audits of algorithms
5. Recognize the limitations of technical fairness solutions

---

## 23.1 Fairness Criteria

### Observational Fairness

- **Demographic parity**: $P(\hat{Y} = 1 | A = 0) = P(\hat{Y} = 1 | A = 1)$
- **Equalized odds**: $P(\hat{Y} = 1 | Y = y, A = 0) = P(\hat{Y} = 1 | Y = y, A = 1)$

### Causal Fairness

- **Counterfactual fairness**: $\hat{Y}_{A \leftarrow a} = \hat{Y}_{A \leftarrow a'}$ for all $a, a'$
- **Path-specific fairness**: No unfair discrimination through prohibited paths

---

## 23.2 Counterfactual Fairness

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm

np.random.seed(42)
n = 1000

# Generate data
A = np.random.binomial(1, 0.5, n)  # Protected attribute
Z = np.random.normal(0, 1, n)  # Sensitive factor
X = 0.5 * A + 0.5 * Z + np.random.normal(0, 1, n)  # Covariates
Y = 2 * X + 0.3 * Z + np.random.normal(0, 1, n)  # Outcome

# Counterfactual fairness test
# For each individual, compare:
# 1. What would Y be if A = 0?
# 2. What would Y be if A = 1?

# If these differ, the decision is not counterfactually fair

# Simplified: check if A directly affects Y
from sklearn.linear_model import LinearRegression

# Model with A
model_with = LinearRegression()
model_with.fit(np.column_stack([X, A]), Y)

# Model without A
model_without = LinearRegression()
model_without.fit(X.reshape(-1, 1), Y)

# If A has a direct effect, counterfactual fairness is violated
print(f"Effect of A on Y (controlling for X): {model_with.coef_[1]:.3f}")
print(f"If this is non-zero, counterfactual fairness may be violated")

---

## 23.3 Path-Specific Fairness

### The Idea

Different causal paths from protected attribute to outcome may be fair or unfair:

- **Direct discrimination**: A → Y (unfair)
- **Indirect discrimination**: A → X → Y (may be fair or unfair)
- **Mediation**: A → Z → Y (depends on Z)

### Implementation

In [ ]:
# Check for direct vs indirect effects
# Direct effect: A → Y
# Indirect effect: A → X → Y

# Mediation analysis to decompose effects
from sklearn.linear_model import LinearRegression

# Total effect
model_total = LinearRegression()
model_total.fit(A.reshape(-1, 1), Y)
total_effect = model_total.coef_[0]

# Direct effect (controlling for X)
model_direct = LinearRegression()
model_direct.fit(np.column_stack([A, X]), Y)
direct_effect = model_direct.coef_[0]

# Indirect effect
indirect_effect = total_effect - direct_effect

print(f"Total effect of A: {total_effect:.3f}")
print(f"Direct effect: {direct_effect:.3f}")
print(f"Indirect effect (through X): {indirect_effect:.3f}")

---

## 23.4 Common Mistakes

1. **Ignoring causal structure**: Observational fairness criteria can be misleading
2. **Proxy discrimination**: Using variables that encode protected attributes
3. **Feedback loops**: Algorithms can amplify existing biases
4. **Simplicity**: Fairness is context-dependent, not just technical

---

## 23.5 Knowledge Check

### Multiple Choice

1. **Counterfactual fairness requires:**
   A) Equal outcomes
   B) Equal treatment
   C) Same prediction regardless of protected attribute
   D) No discrimination

2. **Direct discrimination is:**
   A) A → Y
   B) A → X → Y
   C) X → Y
   D) Y → A

3. **Observational fairness criteria:**
   A) Always imply causal fairness
   B) Never imply causal fairness
   C) May or may not imply causal fairness
   D) Are sufficient for fairness

4. **Proxy discrimination occurs when:**
   A) A variable encodes protected attributes
   B) We measure the protected attribute directly
   C) We ignore the protected attribute
   D) We randomize treatment

5. **Fairness is:**
   A) A purely technical problem
   B) Always possible to achieve
   C) Context-dependent and contested
   D) Defined by algorithms

### Short Answer

6. **Explain the difference between observational and causal fairness.**

7. **What is proxy discrimination and why is it problematic?**

8. **How can causal inference help audit algorithms for bias?**

9. **Why might observational fairness criteria be insufficient?**

10. **Give an example where causal fairness analysis would be valuable.**

---

## 23.6 Summary

1. **Fairness** has both observational and causal definitions
2. **Counterfactual fairness** uses potential outcomes
3. **Path-specific fairness** decomposes effects by causal paths
4. **Causal audits** can identify sources of bias
5. **Context matters**—fairness is not just technical

---

## 23.7 Further Reading

- Kusner, M.J. et al. (2017). "From Parity to Preferences: The Case of Counterfactual Fairness." *NeurIPS*.
- Nabi, R. & Shpitser, I. (2018). "Fair Inference on Outcomes." *AAAI*.